In [ ]:
# Cell 1: Imports and setup
from jupyter_dash import JupyterDash
import dash_leaflet as dl
from dash import dcc, html, dash_table
from dash.dependencies import Input, Output
import plotly.express as px
import base64
import pandas as pd
load_dotenv()

# Import CRUD module
from CRUD_Python_Module import AnimalShelter

# Connect to MongoDB
username = os.getenv("AAC_USER")
password = os.getenv("AAC_PASS")

db = AnimalShelter(username, password)

# Initial data load
df = pd.DataFrame.from_records(db.read({}))
if '_id' in df.columns:
    df.drop(columns=['_id'], inplace=True)

# Cell 2: Logo/branding
image_filename = 'Grazioso_Salvare_Logo.png'  # ensure this file exists in your workspace
encoded_image = base64.b64encode(open(image_filename, 'rb').read()).decode()

logo = html.Img(
    src=f'data:image/png;base64,{encoded_image}',
    style={'height': '100px'},
    id='by Sophie Biondolillo'  # unique identifier
)

# Cell 3: App layout
app = JupyterDash(__name__)

app.layout = html.Div([
    html.Center(html.B(html.H1('CS-340 Grazioso Salvare Dashboard'))),
    logo,
    html.Hr(),

    # Interactive filter options
    dcc.RadioItems(
        id='filter-type',
        options=[
            {'label': 'All Animals', 'value': 'All'},
            {'label': 'Water Rescue', 'value': 'Water Rescue'},
            {'label': 'Mountain or Wilderness Rescue', 'value': 'Mountain or Wilderness Rescue'},
            {'label': 'Disaster or Individual Tracking', 'value': 'Disaster or Individual Tracking'}
        ],
        value='All',
        inline=True,
        style={'margin': '10px 0'}
    ),

    html.Hr(),

    # Interactive DataTable
    dash_table.DataTable(
        id='datatable-id',
        columns=[{"name": i, "id": i, "deletable": False, "selectable": True} for i in df.columns],
        data=df.to_dict('records'),
        filter_action="native",
        sort_action="native",
        sort_mode="multi",
        page_action="native",
        page_current=0,
        page_size=10,
        row_selectable="single",
        style_table={'overflowX': 'auto'},
        style_cell={'textAlign': 'left', 'minWidth': '100px', 'width': '150px', 'maxWidth': '250px'}
    ),

    html.Br(),
    html.Hr(),

    # Charts row
    html.Div(className='row', style={'display': 'flex', 'gap': '20px'}, children=[
        html.Div(id='graph-id', className='col s12 m6'),
        html.Div(id='map-id', className='col s12 m6')
    ])
])

# Cell 4: Filter controller (queries via CRUD)
def rescue_query(filter_type):
    if filter_type == "Water Rescue":
        return {"animal_type": "Dog", "breed": {"$in": ["Labrador Retriever Mix", "Chesapeake Bay Retriever", "Newfoundland"]}}
    elif filter_type == "Mountain or Wilderness Rescue":
        return {"animal_type": "Dog", "breed": {"$in": ["German Shepherd", "Alaskan Malamute", "Old English Sheepdog", "Siberian Husky", "Rottweiler"]}}
    elif filter_type == "Disaster or Individual Tracking":
        return {"animal_type": "Dog", "breed": {"$in": ["Doberman Pinscher", "German Shepherd", "Golden Retriever", "Bloodhound", "Rottweiler"]}}
    else:
        return {}

@app.callback(Output('datatable-id', 'data'),
              [Input('filter-type', 'value')])
def update_dashboard(filter_type):
    query = rescue_query(filter_type)
    data = db.read(query)
    dff = pd.DataFrame.from_records(data)
    if '_id' in dff.columns:
        dff.drop(columns=['_id'], inplace=True)
    return dff.to_dict('records')

# Cell 5: Chart controller
@app.callback(
    Output('graph-id', "children"),
    [Input('datatable-id', "derived_virtual_data")]
)
def update_graphs(viewData):
    dff = pd.DataFrame.from_dict(viewData) if viewData else df
    if dff.empty or 'breed' not in dff.columns:
        fig = px.bar(title='No data available')
    else:
        fig = px.pie(dff, names='breed', title='Distribution of Breeds')
    return [dcc.Graph(figure=fig)]

# Cell 6: Highlight selected columns
@app.callback(
    Output('datatable-id', 'style_data_conditional'),
    [Input('datatable-id', 'selected_columns')]
)
def update_styles(selected_columns):
    selected_columns = selected_columns or []
    return [{
        'if': {'column_id': i},
        'background_color': '#D2F3FF'
    } for i in selected_columns]

# Cell 7: Map controller
@app.callback(
    Output('map-id', "children"),
    [Input('datatable-id', "derived_virtual_data"),
     Input('datatable-id', "derived_virtual_selected_rows")]
)
def update_map(viewData, index):
    if not viewData:
        return []
    dff = pd.DataFrame.from_dict(viewData)
    if index is None or len(index) == 0:
        row = 0
    else:
        row = index[0]

    # Defensive checks for coordinate columns
    # Adjust indices if your dataset uses different column ordering/names
    try:
        lat = dff.iloc[row, 13]
        lon = dff.iloc[row, 14]
        breed = dff.iloc[row, 4]
        name = dff.iloc[row, 9]
    except Exception:
        # Try name-based columns if available
        if {'location_lat', 'location_long'}.issubset(dff.columns):
            lat = dff.loc[dff.index[row], 'location_lat']
            lon = dff.loc[dff.index[row], 'location_long']
        else:
            lat, lon = 30.75, -97.48  # fallback: Austin, TX
        breed = dff.loc[dff.index[row], 'breed'] if 'breed' in dff.columns else 'Unknown'
        name = dff.loc[dff.index[row], 'name'] if 'name' in dff.columns else 'Unknown'

    return [
        dl.Map(style={'width': '100%', 'height': '500px'}, center=[30.75, -97.48], zoom=10, children=[
            dl.TileLayer(id="base-layer-id"),
            dl.Marker(position=[lat, lon], children=[
                dl.Tooltip(str(breed)),
                dl.Popup([html.H1("Animal Name"), html.P(str(name))])
            ])
        ])
    ]

# Cell 8: Run app
app.run_server(host='localhost', port=8050)